# Part 4 · Notebook 02 — Contracts, symbols and accounts

**Sessions:** S3 (`ib_async` core) · S4 (Alpaca core) · [Lesson plan](../../docs/lessons/PART_04_BROKER_CONNECTIVITY.md) · graded labs in [`labs/part04/`](../../labs/part04/)

**You will:**
1. Give every instrument one canonical symbol, whichever broker it came from.
2. See what an IB contract needs for stocks, futures, FX and crypto.
3. Turn IB's string account rows into exact `Decimal` numbers.
4. Read an Alpaca account for the warnings that should stop a strategy from starting.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Nothing here connects to a broker: the account rows, bars, ticks and order events are synthetic, shaped like what `ib_async` and `alpaca-py` return.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p4lib.py is in notebooks/part04/
    sys.path.insert(0, str(d))
from decimal import Decimal
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p4lib as p

p.use_course_style()

## 1. One symbol per instrument

Each broker names things its own way (`EUR.USD` vs `EUR/USD`, `ESZ6` vs `ES 202612`). Our data store and order book use one **canonical** symbol:

| Asset class | Example | Rule |
|---|---|---|
| Equity | `AAPL` | the ticker |
| Future | `ESZ6` | root + month code + last digit of the year (expiry `YYYYMM`) |
| FX | `EUR.USD` | base `.` quote |
| Crypto | `BTC/USD` | base `/` currency |

Futures month codes, January to December: `F G H J K M N Q U V X Z` (in `p.MONTH_CODES`).

In [ ]:
pd.DataFrame([vars(i) for i in p.INSTRUMENTS])

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def canonical_symbol(inst: p.Instrument) -> str:
    match inst.asset_class:
        case "EQUITY":
            return inst.symbol
        case "FUTURE":
            return ...                            # ✍️ root + p.MONTH_CODES[month - 1] + last digit of the year
        case "FX":
            return f"{inst.symbol[:3]}.{inst.symbol[3:]}"
        case "CRYPTO":
            return ...                            # ✍️ base "/" currency
    raise ValueError(inst.asset_class)

mine = [canonical_symbol(i) for i in p.INSTRUMENTS]
mine = p.check("canonical symbols", mine, [p.canonical_symbol(i) for i in p.INSTRUMENTS])
mine

## 2. What IB needs to identify a contract

`ib_async` builds these with `Stock(...)`, `Future(...)`, `Forex(...)`, `Crypto(...)`. The fields that trip people up:
* stocks route via `SMART` but need `primaryExchange` to be unambiguous;
* futures need the contract month, and you **trade** a specific month (`ContFuture` is for data only);
* FX is `CASH` on `IDEALPRO`, with the *quote* currency as `currency`.

In [ ]:
pd.DataFrame({p.canonical_symbol(i): p.ib_contract_fields(i) for i in p.INSTRUMENTS}).T.fillna("")

## 3. IB account values are strings, in several currencies

`ib.accountValues()` returns rows like `AccountValue(account, tag, value, currency, modelCode)`. Every value is a **string**, the same tag appears once per currency plus a `BASE` line, and many tags aren't numbers at all.

In [ ]:
rows = p.ib_account_rows()
pd.DataFrame(rows)

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

Keep the rows whose tag is in `p.SUMMARY_TAGS` **and** whose currency matches; convert the value with `Decimal`.

In [ ]:
def account_summary(rows: list[dict], currency: str = "USD") -> dict[str, Decimal]:
    return ...                                    # ✍️ {tag: Decimal(value)} for the summary tags in `currency`

mine = account_summary(rows)
mine = p.check("account summary", mine, p.ib_account_summary(rows))
mine

In [ ]:
naive = {r["tag"]: float(r["value"]) for r in rows if r["tag"] in p.SUMMARY_TAGS}   # no currency filter
print("NetLiquidation, naive:", naive["NetLiquidation"], " ← the EUR row came last and overwrote the USD value")
print("NetLiquidation, right:", p.ib_account_summary(rows)["NetLiquidation"])

## 4. Alpaca: read the account before you trade

`TradingClient.get_account()` returns the account with flags that must stop a strategy at startup:
* `status` other than `ACTIVE`, `account_blocked`, `trading_blocked`;
* the **pattern day trader** rule: flagged as PDT with equity under $25,000 means no more day trades;
* buying power at or below zero.

Return the warnings **in that order**: `'status <status>'`, `'account blocked'`, `'trading blocked'`, `'PDT with equity below $25,000'`, `'no buying power'`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def account_warnings(acct: dict) -> list[str]:
    out = []
    if acct["status"] != "ACTIVE":
        out.append(f"status {acct['status']}")
    if acct["account_blocked"]:
        out.append("account blocked")
    if acct["trading_blocked"]:
        out.append("trading blocked")
    if ...:                                       # ✍️ PDT flag and equity (a string!) below 25,000
        out.append("PDT with equity below $25,000")
    if ...:                                       # ✍️ buying power at or below zero
        out.append("no buying power")
    return out

mine = {name: account_warnings(a) for name, a in p.ALPACA_ACCOUNTS.items()}
mine = p.check("Alpaca account warnings", mine, {name: p.alpaca_account_warnings(a) for name, a in p.ALPACA_ACCOUNTS.items()})
mine

## 5. Both accounts in one table

This is Exercise 1 of the lesson plan, on synthetic data. In Clinic W1 you run the same table against your real paper accounts with `labs/part04/paper/connect_both.py`.

In [ ]:
ib = p.ib_account_summary(rows)
alp = p.ALPACA_ACCOUNTS["healthy"]
table = pd.DataFrame({
    "IB (DU…567)": {"equity": ib["NetLiquidation"], "cash": ib["TotalCashValue"], "buying power": ib["BuyingPower"]},
    "Alpaca": {"equity": Decimal(alp["equity"]), "cash": Decimal(alp["cash"]), "buying power": Decimal(alp["buying_power"])},
})
table.map(lambda d: f"${d:,.2f}")

## Wrap-up

* Canonical symbols let data and orders from both brokers meet in one place.
* Account numbers arrive as strings: parse them to `Decimal`, and filter by currency.
* Check account flags **before** the first order, not after the first rejection.
* Graded version: `labs/part04/week13_foundations` (`canonical_symbol`, `to_ib_contract` with real `ib_async` objects, `ib_account_summary`, `alpaca_account_warnings`).